# **EXTRAÇÃO POR LLM**
**Nome:** Gisela Kassick e Lívia Aragão

## Importações

In [ ]:
import os
import json
import re
import time

import pandas as pd
from openai import OpenAI

## Configurações

In [ ]:
ARQUIVO_ENTRADA = "extracao_entrada.xlsx"
ARQUIVO_SAIDA = "extracaollm.xlsx"
ARQUIVO_CHECKPOINT = "extracao_checkpoints.csv"

BASE_URL = "https://iluma.cnpem.br:4000/v1"
MODELO = "iluma"

TEMP_EXTRACAO = 0.5

MAX_TOKENS_INICIAL = 1500
MAX_TOKENS_MAXIMO = 6000
FATOR_AUMENTO_TOKENS = 2

MAX_TENTATIVAS = 6
TIMEOUT = 6000.0

SALVAR_A_CADA = 100

token = "SEU_TOKEN"

if not token:
    raise RuntimeError("A chave da API nao foi encontrada.")

cliente = OpenAI(base_url=BASE_URL, api_key=token, timeout=TIMEOUT, max_retries=1)

## Prompt

In [ ]:
SISTEMA_EXTRACAO = """Extraia o material que constitui o eletrodo (electrode) a partir de
uma janela de contexto de 100 tokens em torno da palavra "electrode".

Regras:
1. Extraia APENAS o material do eletrodo. Ignore reagentes ou materiais envolvidos
   na síntese/teste do dispositivo, mas que não compõem o eletrodo em si.
2. Os materiais não precisam estar em forma de fórmula química.
3. Se o eletrodo tiver base + revestimento/funcionalização (ex.: "eletrodo de carbono
   recoberto com MoS2"), retorne uma entrada separada em "extracoes" para cada material.
4. Se não for possível extrair o material dessa janela de contexto, retorne
   {"extracoes": []}.

Responda APENAS com um JSON no formato:
{"extracoes": [{"material_extraction": "..."}]}
Sem texto adicional, sem markdown, sem explicações.
"""

## Limpeza

In [ ]:
def limpar_cercas(texto):
    
    if not isinstance(texto, str):
        return ""

    texto = texto.strip()

    if texto.startswith("```"):
        texto = re.sub(r"^```(?:json)?\s*", "", texto, flags=re.IGNORECASE)
        texto = re.sub(r"\s*```$", "", texto)

    return texto.strip()

## Extrair json de uma resposta

In [ ]:
def extrair_json(texto):
    
    texto = limpar_cercas(texto)
    
    if not texto:
        return None
    try:
        return json.loads(texto) 
    except json.JSONDecodeError:
        pass

    inicio = texto.find("{")
    fim = texto.rfind("}")
    
    if inicio >= 0 and fim > inicio:
        trecho = texto[inicio:fim + 1]
    
        try:
            return json.loads(trecho)
        except json.JSONDecodeError:
            pass
    
    return None

## Chamada ao modelo

In [ ]:
def perguntar(pergunta, sistema=None, **kw):
    mensagens = []
    
    if sistema:
        mensagens.append({"role": "system", "content": sistema})
    
    mensagens.append({"role": "user", "content": pergunta})
    
    resposta = cliente.chat.completions.create(model=MODELO, messages=mensagens, **kw)
    
    if not resposta.choices:
        raise RuntimeError("A API retornou zero choices.")
    
    escolha = resposta.choices[0]
    finish_reason = getattr(escolha, "finish_reason", None)
    mensagem = escolha.message
    conteudo = getattr(mensagem, "content", None)
    reasoning = getattr(mensagem, "reasoning_content", None)
    
    if conteudo is None:
        extra = ""
    
        if reasoning:
            extra = f" | reasoning_content presente ({len(str(reasoning))} chars) - provavel estouro de max_tokens em raciocinio"
    
        print(f"DEBUG | finish_reason={finish_reason} | content=None{extra}", flush=True)
    
    elif not str(conteudo).strip():
        print(f"DEBUG | finish_reason={finish_reason} | content vazio", flush=True)
    
    return conteudo, finish_reason

## Identificar erros temporários

In [ ]:
def erro_temporario(e):
    status = getattr(e, "status_code", None)
    
    if status in {408, 409, 429, 500, 502, 503, 504}:
        return True
    
    texto = str(e).lower()
    
    termos = ["timeout", "timed out", "connection", "temporarily unavailable", "upstream error", "server error", "rate limit"]
    
    return any(termo in texto for termo in termos)

## Extração de um contexto

In [ ]:
def extrair_llm(sentenca):
    if not isinstance(sentenca, str):
        return [], "Contexto vazio"
    
    if not sentenca.strip():
        return [], "Contexto vazio"
    
    ultimo_motivo = None
    max_tokens_atual = MAX_TOKENS_INICIAL
    
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            bruto, finish_reason = perguntar(sentenca, sistema=SISTEMA_EXTRACAO, temperature=TEMP_EXTRACAO, max_tokens=max_tokens_atual)
    
            if bruto is None or not str(bruto).strip():
                ultimo_motivo = f"Resposta vazia (finish_reason={finish_reason}, max_tokens={max_tokens_atual})"
    
                if tentativa < MAX_TENTATIVAS:
                    espera = 2 ** (tentativa - 1)
    
                    if finish_reason == "length":
                        max_tokens_atual = min(max_tokens_atual * FATOR_AUMENTO_TOKENS, MAX_TOKENS_MAXIMO)
    
                    print(f"Resposta vazia. Tentativa {tentativa}/{MAX_TENTATIVAS}. Proximo max_tokens={max_tokens_atual}. Nova tentativa em {espera}s.", flush=True)
    
                    time.sleep(espera)
                    continue
    
                return [], ultimo_motivo
    
            dados = extrair_json(bruto)
    
            if dados is None:
                ultimo_motivo = f"JSON invalido: {str(bruto)[:200]}"
    
                if tentativa < MAX_TENTATIVAS:
                    espera = 2 ** (tentativa - 1)
    
                    print(f"JSON invalido. Tentativa {tentativa}/{MAX_TENTATIVAS}. Nova tentativa em {espera}s.", flush=True)
    
                    time.sleep(espera)
                    continue
    
                return [], ultimo_motivo
    
            if not isinstance(dados, dict):
                return [], "JSON nao possui formato de objeto"
    
            extracoes = dados.get("extracoes", [])
    
            if extracoes is None:
                extracoes = []
    
            if not isinstance(extracoes, list):
                return [], "Campo extracoes nao e uma lista"
    
            if not extracoes:
                return [], "Extracao nao encontrada"
    
            extracoes_validas = []
    
            for ext in extracoes:
                if not isinstance(ext, dict):
                    continue
    
                material = ext.get("material_extraction")
    
                if not isinstance(material, str):
                    continue
    
                material = material.strip()
    
                if not material:
                    continue
    
                extracoes_validas.append({"material_extraction": material})
    
            if not extracoes_validas:
                return [], "Extracao nao encontrada"
    
            return extracoes_validas, None
    
        except Exception as e:
            ultimo_motivo = f"Erro de API: {e}"
    
            if erro_temporario(e) and tentativa < MAX_TENTATIVAS:
                espera = 2 ** (tentativa - 1)
    
                print(f"Erro temporario da API. Tentativa {tentativa}/{MAX_TENTATIVAS}. Nova tentativa em {espera}s.", flush=True)
    
                time.sleep(espera)
                continue
    
            return [], ultimo_motivo
    
    return [], ultimo_motivo or "Erro desconhecido"


## Adicionar checkpoints por segurança

In [ ]:
def carregar_checkpoint(df, checkpoint_path):
    total = len(df)
    materiais = [None] * total
    motivos = [None] * total
    
    if not os.path.exists(checkpoint_path):
        print("Nenhum checkpoint encontrado.")
        return materiais, motivos
    
    print(f"Checkpoint encontrado: {checkpoint_path}")
    
    try:
        checkpoint = pd.read_csv(checkpoint_path)
    except Exception as e:
        print(f"Nao foi possivel carregar o checkpoint: {e}")
        return materiais, motivos
    
    if "material_electrode" not in checkpoint.columns or "motivo_falha" not in checkpoint.columns:
        print("Checkpoint nao possui as colunas esperadas.")
        return materiais, motivos
    
    quantidade = min(total, len(checkpoint))
    
    for i in range(quantidade):
        valor_material = checkpoint.iloc[i]["material_electrode"]
        valor_motivo = checkpoint.iloc[i]["motivo_falha"]
    
        if not pd.isna(valor_material):
            materiais[i] = str(valor_material)
    
        if not pd.isna(valor_motivo):
            motivos[i] = str(valor_motivo)
    
    print(f"Checkpoint carregado com {quantidade} registros.")
    
    return materiais, motivos

def registro_processado(material, motivo):
    if material is not None and not pd.isna(material) and str(material).strip():
    return True
    
    if motivo is not None and not pd.isna(motivo) and str(motivo).strip():
        if str(motivo).startswith("Resposta vazia"):
            return False
    
        return True
    
    return False

def salvar_checkpoint(df, materiais, motivos, checkpoint_path):
    df_checkpoint = df.copy()
    df_checkpoint["material_electrode"] = materiais
    df_checkpoint["motivo_falha"] = motivos
    df_checkpoint.to_csv(checkpoint_path, index=False)


## Execução

In [ ]:
def extrair_llm_dataset(df, coluna, coluna_titulo="Title", coluna_abstract="Abstract", checkpoint_path=ARQUIVO_CHECKPOINT, salvar_a_cada=SALVAR_A_CADA):
    df = df.copy()
    total = len(df)
    
    materiais, motivos = carregar_checkpoint(df, checkpoint_path)
    
    print()
    print("INICIO DA EXTRACAO")
    print(f"Total de registros: {total}")
    print()
    
    processados = sum(registro_processado(materiais[i], motivos[i]) for i in range(total))
    
    print(f"Registros ja processados: {processados}/{total}")
    print(f"Registros restantes: {total - processados}")
    print()
    
    novos_processados = 0
    
    for i in range(total):
        if registro_processado(materiais[i], motivos[i]):
            print(f"[{i + 1}/{total}] Ja processado - pulando", flush=True)
            continue
    
        sentenca = df.iloc[i][coluna]
        extracoes, motivo = extrair_llm(sentenca)
    
        lista = []
    
        for ext in extracoes:
            if not isinstance(ext, dict):
                continue
    
            material = ext.get("material_extraction")
    
            if isinstance(material, str) and material.strip():
                lista.append(material.strip())
    
        lista = list(dict.fromkeys(lista))
        material = ", ".join(lista) if lista else None
    
        materiais[i] = material
        motivos[i] = motivo
    
        if material:
            print(f"[{i + 1}/{total}] Artigo analisado | Material: {material}", flush=True)
        elif motivo:
            print(f"[{i + 1}/{total}] Artigo analisado | Resultado: {motivo}", flush=True)
        else:
            print(f"[{i + 1}/{total}] Artigo analisado | Nenhum material identificado", flush=True)
    
        novos_processados += 1
    
        if salvar_a_cada and novos_processados % salvar_a_cada == 0:
            salvar_checkpoint(df, materiais, motivos, checkpoint_path)
            print()
            print(f"> CHECKPOINT SALVO: {i + 1}/{total} registros")
            print()
    
    salvar_checkpoint(df, materiais, motivos, checkpoint_path)
    
    df["material_electrode"] = materiais
    df["motivo_falha"] = motivos
    
    revisar = df[df["motivo_falha"] == "Extracao nao encontrada"][[coluna_titulo, coluna_abstract, coluna]]
    
    erros_tecnicos = df[df["motivo_falha"].notna() & (df["motivo_falha"] != "Extracao nao encontrada")]
    
    print()
    print("EXTRACAO FINALIZADA")
    print(f"Total de artigos: {len(df)}")
    print(f"Materiais encontrados: {df['material_electrode'].notna().sum()}")
    print(f"Sem material encontrado: {len(revisar)}")
    print(f"Falhas tecnicas: {len(erros_tecnicos)}")
    
    return df, revisar, erros_tecnicos

print()
print("Carregando arquivo de entrada...")

df_electrode = pd.read_excel(ARQUIVO_ENTRADA)

print(f"Arquivo carregado: {ARQUIVO_ENTRADA}")
print(f"Numero de registros: {len(df_electrode)}")

COLUNA_EXTRACAO = "contexto_electrode"

if COLUNA_EXTRACAO not in df_electrode.columns:
    raise ValueError(f"A coluna '{COLUNA_EXTRACAO}' nao foi encontrada no arquivo.\n\nColunas disponiveis:\n{list(df_electrode.columns)}")

df_extraction, revisar, erros_tecnicos = extrair_llm_dataset(df_electrode, COLUNA_EXTRACAO, checkpoint_path=ARQUIVO_CHECKPOINT, salvar_a_cada=SALVAR_A_CADA)

print()
print("Salvando resultado final...")

df_extraction.to_excel(ARQUIVO_SAIDA, index=False, engine="openpyxl")

print()
print("ARQUIVO FINAL SALVO")
print(f"Arquivo: {ARQUIVO_SAIDA}")
